# EDA dữ liệu cảm xúc VLSP — khảo sát trước tiền xử lý

Dữ liệu Base/Augmented được phân tích ở dạng gốc. Test chỉ dùng kiểm toán schema và giao tập, không dùng nhãn test để chọn quy tắc. Mọi tỷ lệ theo nhãn dưới đây chỉ dùng tập phát triển.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub>=0.3,<2', 'transformers>=4.40,<5', 'sentencepiece', 'pyvi==0.1.1', 'py_vncorenlp', 'gensim>=4.4,<5'])


In [ ]:
import re, unicodedata, json, hashlib
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
LABELS = [-1, 0, 1]
DATA_DIR = None
DATASET_HANDLE = 'anhphmduy/vlsp-sentiment-dataset/versions/5'
OUT = Path('vlsp_raw_eda'); OUT.mkdir(exist_ok=True)

In [ ]:
FILES = {
    "Base": ("vlsp_sentiment_train.csv",),
    "Augmented": ("vlsp_sentiment_train_augmented_v2.csv", "augmented_v2.csv"),
    "Test": ("vlsp_sentiment_test.csv",),
}

def locate_inputs(root):
    return {
        name: next((root / filename for filename in filenames if (root / filename).is_file()), None)
        for name, filenames in FILES.items()
    }

def resolve_inputs():
    if DATA_DIR is not None:
        root = Path(DATA_DIR).resolve()
    else:
        candidates = [
            Path("data"), Path("."),
            Path("/kaggle/input/vlsp-sentiment-dataset"),
            Path("/kaggle/input/datasets/anhphmduy/vlsp-sentiment-dataset"),
        ]
        root = next((p for p in candidates if all(locate_inputs(p).values())), None)
        if root is None:
            import kagglehub
            root = Path(kagglehub.dataset_download(DATASET_HANDLE))
            root = next(
                (p.parent for p in sorted(root.rglob(FILES["Base"][0]))
                 if all(locate_inputs(p.parent).values())),
                root,
            )
        root = root.resolve()
    paths = locate_inputs(root)
    missing = ["/".join(FILES[name]) for name, path in paths.items() if path is None]
    if missing:
        raise FileNotFoundError(f"Thiếu dữ liệu VLSP tại {root}: {', '.join(missing)}")
    return root, paths

def read_vlsp(path, split):
    for delimiter in ("\t", ","):
        header = pd.read_csv(path, sep=delimiter, encoding="utf-8-sig", nrows=0)
        if {"Class", "Data"}.issubset(header.columns.str.strip()):
            break
    else:
        raise ValueError(f"{path.name} cần hai cột Class và Data, phân tách bằng tab hoặc dấu phẩy.")
    frame = pd.read_csv(path, sep=delimiter, encoding="utf-8-sig", keep_default_na=False, dtype=str)
    frame.columns = frame.columns.str.strip()
    frame = frame[["Class", "Data"]].copy()
    labels = pd.to_numeric(frame["Class"], errors="coerce")
    invalid = ~labels.isin(LABELS)
    if frame.empty or invalid.any():
        raise ValueError(f"{path.name}: {int(invalid.sum())} nhãn sai hoặc tập rỗng.")
    frame["label"] = labels.astype(int)
    frame = frame.rename(columns={"Data": "text"})
    frame["row"] = np.arange(len(frame))
    frame["sample_id"] = split.lower() + "-" + frame["row"].astype(str)
    return frame[["sample_id", "row", "label", "text"]]

data_root, paths = resolve_inputs()
raw = {name: read_vlsp(path, name) for name, path in paths.items()}
print({name: len(frame) for name, frame in raw.items()})

## Phân bố, độ dài và chất lượng dữ liệu gốc

Tỷ lệ theo lớp dùng Base và Augmented. Giao test chỉ dựa trên văn bản, không dùng nhãn test.

In [ ]:
dev = pd.concat([raw['Base'].assign(dataset='Base'), raw['Augmented'].assign(dataset='Augmented')], ignore_index=True)
counts = dev.groupby(['dataset','label']).size().unstack(fill_value=0).reindex(columns=LABELS,fill_value=0)
display(counts)
counts.to_csv(OUT/'label_counts.csv')
counts.plot.bar(rot=0, title='Số câu gốc theo nhãn', ylabel='Số câu'); plt.tight_layout(); plt.savefig(OUT/'labels.png'); plt.show()
lengths = dev.assign(chars=dev.text.str.len(), words=dev.text.str.split().str.len())
display(lengths.groupby(['dataset','label'])[['chars','words']].quantile([.5,.9,.95,.99,1]).round(1))
fig, axes=plt.subplots(1,2,figsize=(12,4))
for name, part in lengths.groupby('dataset'):
 for ax,col in zip(axes,['chars','words']):
  v=np.sort(part[col]); ax.step(v,100*np.arange(1,len(v)+1)/len(v),label=name)
for ax,col in zip(axes,['Ký tự','Đơn vị cách trắng']): ax.set(xlabel=col,ylabel='% câu không vượt ngưỡng'); ax.legend()
fig.tight_layout(); fig.savefig(OUT/'lengths.png'); plt.show()

In [ ]:
def raw_key(text): return ' '.join(unicodedata.normalize('NFC',text).casefold().split())
keys={name:frame.text.map(raw_key) for name,frame in raw.items()}
audit=[]
for name,frame in raw.items():
 key=keys[name]; nonblank=key.ne('')
 conflicts=pd.DataFrame({'key':key,'label':frame.label}).loc[nonblank].groupby('key').label.nunique().gt(1).sum() if name!='Test' else None
 audit.append(dict(dataset=name,rows=len(frame),blank=int((~nonblank).sum()),duplicate_extras=int(key[nonblank].duplicated().sum()),conflicting_keys=conflicts,overlap_test=int((nonblank & key.isin(keys['Test'])).sum()) if name!='Test' else None))
audit=pd.DataFrame(audit); display(audit); audit.to_csv(OUT/'raw_audit.csv',index=False)
audit.set_index('dataset')[['blank','duplicate_extras','overlap_test']].plot.bar(rot=0,title='Kiểm toán gốc: nhóm có thể chồng lấn'); plt.tight_layout(); plt.savefig(OUT/'audit.png'); plt.show()

## Tín hiệu trong văn bản gốc

Tỷ lệ là phần trăm câu có ít nhất một tín hiệu; đồng xuất hiện không chứng minh ý nghĩa hay quan hệ nhân quả. Ví dụ lấy theo sample_id để có thể kiểm tra ngữ cảnh.

In [ ]:
patterns={'phủ định':r'(?i)(?<!\w)(không|chẳng|chưa|ko|hok|khg)(?!\w)', 'teencode ứng viên':r'(?i)(?<!\w)(ko|hok|khg|đc|dc|sp|đt|thik|iu|mik|vs|bthg|cx)(?!\w)', 'kéo dài':r'(?i)([^\W\d_])\1{2,}', 'emoticon':r'(?i)(?::[-]?[)(d]|\^\^|t\.t)', 'emoji':r'[\U0001F300-\U0001FAFF\u2764\uFE0F]'}
rows=[]
for name,pat in patterns.items():
 mask=dev.text.str.contains(pat,regex=True,na=False)
 for (dataset,label), group in dev.groupby(['dataset','label']):
  subset=mask.loc[group.index]; rows.append({'signal':name,'dataset':dataset,'label':label,'count':int(subset.sum()),'total':len(group),'percent':100*subset.mean()})
 print('\n',name,'— ví dụ gốc'); display(dev.loc[mask,['dataset','sample_id','label','text']].sort_values(['dataset','sample_id']).head(8))
signals=pd.DataFrame(rows); display(signals); signals.to_csv(OUT/'signal_rates.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(14,5),sharey=True)
for ax,name in zip(axes,['Base','Augmented']):
 signals[signals.dataset==name].pivot(index='signal',columns='label',values='percent').plot.barh(ax=ax); ax.set(title=name,xlabel='% câu có tín hiệu',ylabel='')
fig.tight_layout(); fig.savefig(OUT/'signals.png'); plt.show()

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
for name,frame in dev.groupby('dataset'):
 vectorizer=CountVectorizer(ngram_range=(1,2),binary=True,token_pattern=r'(?u)\b\w+\b',min_df=3,max_features=20000)
 matrix=vectorizer.fit_transform(frame.text); terms=vectorizer.get_feature_names_out()
 for label in LABELS:
  idx=np.flatnonzero(frame.label.to_numpy()==label); freq=np.asarray(matrix[idx].sum(axis=0)).ravel(); top=np.argsort(-freq,kind='stable')[:15]
  display(pd.DataFrame({'dataset':name,'label':label,'term':terms[top],'document_count':freq[top]}))

## Đối chiếu từng quy tắc trên cùng dữ liệu gốc


In [ ]:
# Khảo sát các quy tắc độc lập; chưa sử dụng chúng để sửa dữ liệu nguồn.
import os, shutil, hashlib
from collections import Counter
from itertools import groupby
from functools import lru_cache
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoTokenizer
from pyvi import ViTokenizer
import py_vncorenlp

def minimal(text):
    return ' '.join(unicodedata.normalize('NFC', text).split())

TEENCODE = {'ko':'không','hok':'không','khg':'không','đc':'được','dc':'được','sp':'sản phẩm','đt':'điện thoại','thik':'thích','iu':'yêu','mik':'mình','vs':'với','bthg':'bình thường','cx':'cũng'}
EMOTICONS = {'❤️':'yêu thích','👍':'tốt','😊':'vui vẻ','😍':'yêu thích','👎':'chê','😡':'tức giận','😭':'khóc','💔':'thất vọng',':)':'cười',':-)':'cười',':(':'buồn',':-(':'buồn',':d':'cười','^^':'vui','t.t':'khóc'}
teen_re = re.compile(r'\b(?:'+'|'.join(map(re.escape,TEENCODE))+r')\b',re.I)
emote_re = re.compile('|'.join(map(re.escape,sorted(EMOTICONS,key=len,reverse=True))),re.I)

def teen(text): return teen_re.sub(lambda m:TEENCODE[m[0].lower()],text)
def emote(text): return emote_re.sub(lambda m:' '+EMOTICONS[m[0].lower()]+' ',text)
def elong(text): return re.sub(r'([^\W\d_])\1{2,}',r'\1',text,flags=re.I)
def strip_symbols(text): return ' '.join(re.sub(r'[^\w\s.,!?]',' ',text).split())
transforms={'NFC+khoảng trắng':minimal,'+chữ thường':lambda x:minimal(x).lower(),'+teencode':lambda x:teen(minimal(x)), '+emoticon thành từ':lambda x:emote(minimal(x)), '+rút chữ kéo dài':lambda x:elong(minimal(x)), '+lọc ký hiệu':lambda x:strip_symbols(minimal(x))}
rule_rows=[]; examples=[]; collision_rows=[]
for dataset, frame in dev.groupby('dataset'):
    original=frame.text
    for name,fn in transforms.items():
        changed=original.map(fn)
        groups=pd.DataFrame({'raw':original,'processed':changed,'label':frame.label}).groupby('processed')
        rule_rows.append(dict(dataset=dataset,rule=name,changed=int(changed.ne(original).sum()),percent=100*changed.ne(original).mean(),characters_removed=int((original.str.len()-changed.str.len()).clip(lower=0).sum()),empty=int(changed.str.strip().eq('').sum()),collision_groups=int(groups.raw.nunique().gt(1).sum()),conflicting_groups=int(groups.label.nunique().gt(1).sum())))
        for idx in frame.loc[changed.ne(original)].sort_values('sample_id').head(5).index:
            examples.append(dict(dataset=dataset,rule=name,sample_id=dev.loc[idx,'sample_id'],original=original.loc[idx],processed=changed.loc[idx]))
        for processed,group in groups:
            if group.raw.nunique()>1:
                collision_rows.extend(dict(dataset=dataset,rule=name,original=r.raw,processed=processed,label=r.label) for r in group.itertuples())
rules=pd.DataFrame(rule_rows); display(rules); rules.to_csv(OUT/'rule_impacts.csv',index=False)
pd.DataFrame(examples).to_csv(OUT/'rule_examples.csv',index=False); display(pd.DataFrame(examples))
pd.DataFrame(collision_rows,columns=['dataset','rule','original','processed','label']).to_csv(OUT/'rule_collisions.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(14,5))
for ax,dataset in zip(axes,['Base','Augmented']):
    part=rules[rules.dataset==dataset]; ax.barh(part.rule,part.percent); ax.set(title=dataset,xlabel='% dòng thay đổi')
fig.tight_layout(); fig.savefig(OUT/'rule-impacts.png',dpi=140); plt.show(); plt.close(fig)
# Từ viết tắt được đếm từng biểu thức, kèm ngữ cảnh để tránh suy nghĩa từ tần suất.
term_examples=[]
for term in TEENCODE:
    mask=dev.text.str.contains(r'(?i)(?<!\w)'+re.escape(term)+r'(?!\w)',regex=True)
    for row in dev.loc[mask].sort_values(['dataset','sample_id']).head(6).itertuples():
        term_examples.append(dict(term=term,dataset=row.dataset,sample_id=row.sample_id,label=row.label,text=row.text))
display(pd.DataFrame(term_examples)); pd.DataFrame(term_examples).to_csv(OUT/'teencode_contexts.csv',index=False)

# Giữ bản gốc và tạo các biểu diễn ứng viên riêng.
for name,frame in raw.items():
    frame['minimal']=frame.text.map(minimal)
    frame['tfidf_text']=frame.minimal.str.lower()
working=Path.cwd(); segdir=Path('vncorenlp').resolve(); segdir.mkdir(exist_ok=True)
try:
    if not (segdir/'VnCoreNLP-1.2.jar').exists():py_vncorenlp.download_model(save_dir=str(segdir))
    segmenter=py_vncorenlp.VnCoreNLP(annotators=['wseg'],save_dir=str(segdir))
finally:os.chdir(working)
@lru_cache(maxsize=None)
def segment(text):return ' '.join(segmenter.word_segment(text))
for dataset in ['Base','Augmented']:
    frame=raw[dataset]
    frame['cbow_text']=frame.minimal.str.lower().map(ViTokenizer.tokenize)
    frame['phobert_text']=frame.minimal.map(segment)
tokenizer=AutoTokenizer.from_pretrained('vinai/phobert-base-v2',use_fast=False)
print('Đã tạo ba biểu diễn riêng; chưa huấn luyện classifier và không dùng nhãn test.')


## Độ phủ TF-IDF, CBOW và mức cắt PhoBERT


In [ ]:
# TF-IDF: chỉ fit trên tập phát triển từng phiên bản, không dùng test.
WORD_PATTERN=r'(?u)\b\w+\b'
SYMBOL_PATTERN=r'(?u)(?::[-]?[)(d]|\^\^|t\.t|\b\w+\b)'
tfidf_rows=[]; coverage_rows=[]; dfs=[]
for dataset in ['Base','Augmented']:
    frame=raw[dataset]; texts=frame.tfidf_text
    for analyzer_name,pattern in [('word',WORD_PATTERN),('word+emoticon',SYMBOL_PATTERN)]:
        broad=TfidfVectorizer(token_pattern=pattern,ngram_range=(1,2),lowercase=False,dtype=np.float32)
        broad_matrix=broad.fit_transform(texts); total_occurrences=broad_matrix.getnnz()
        docfreq=np.asarray((broad_matrix>0).sum(axis=0)).ravel()
        dfs.extend(dict(dataset=dataset,analyzer=analyzer_name,df=int(value)) for value in docfreq)
        for min_df in [1,2,3,5]:
            for cap in [5000,10000,20000]:
                vectorizer=TfidfVectorizer(token_pattern=pattern,ngram_range=(1,2),lowercase=False,min_df=min_df,max_features=cap,sublinear_tf=True,dtype=np.float32)
                matrix=vectorizer.fit_transform(texts)
                empty=np.asarray(matrix.getnnz(axis=1)==0)
                tfidf_rows.append(dict(dataset=dataset,analyzer=analyzer_name,min_df=min_df,max_features=cap,features=matrix.shape[1],nnz=matrix.nnz,sparsity=1-matrix.nnz/(matrix.shape[0]*matrix.shape[1]),feature_occurrence_coverage=matrix.nnz/total_occurrences,empty_documents=int(empty.sum()),sparse_bytes=matrix.data.nbytes+matrix.indices.nbytes+matrix.indptr.nbytes,dense_batch128_bytes=128*matrix.shape[1]*4))
        # Mức thay đổi biểu diễn khi áp dụng từng quy tắc; cùng analyzer thực tế.
        for name,fn in transforms.items():
            values=frame.text.map(fn).str.lower()
            candidate=TfidfVectorizer(token_pattern=pattern,ngram_range=(1,2),max_features=5000,sublinear_tf=True,lowercase=False,dtype=np.float32)
            m=candidate.fit_transform(values)
            coverage_rows.append(dict(dataset=dataset,rule=name,analyzer=analyzer_name,features=m.shape[1],nnz=m.nnz,empty_documents=int((m.getnnz(axis=1)==0).sum())))
tfidf=pd.DataFrame(tfidf_rows); tfidf.to_csv(OUT/'tfidf_diagnostics.csv',index=False); display(tfidf)
pd.DataFrame(coverage_rows).to_csv(OUT/'tfidf_rule_comparison.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(12,4),sharey=True)
for ax,dataset in zip(axes,['Base','Augmented']):
    for name,group in tfidf[(tfidf.dataset==dataset)&(tfidf.min_df==1)].groupby('analyzer'):
        ax.plot(group.max_features,100*group.feature_occurrence_coverage,marker='o',label=name)
    ax.set(title=dataset,xlabel='Giới hạn số đặc trưng',ylabel='% lượt đặc trưng-văn bản được giữ'); ax.legend()
fig.tight_layout(); fig.savefig(OUT/'tfidf-coverage.png',dpi=140); plt.show(); plt.close(fig)
fig,axes=plt.subplots(1,2,figsize=(12,4))
for ax,dataset in zip(axes,['Base','Augmented']):
    values=pd.DataFrame(dfs); values=values[(values.dataset==dataset)&(values.analyzer=='word')].df.to_numpy(); values=np.sort(values)
    ax.step(values,100*np.arange(1,len(values)+1)/len(values)); ax.set(xscale='log',xlabel='Document frequency',ylabel='% đặc trưng có DF không vượt ngưỡng',title=dataset)
fig.tight_layout(); fig.savefig(OUT/'tfidf-df.png',dpi=140); plt.show(); plt.close(fig)

from gensim.models import KeyedVectors
vectors=KeyedVectors.load_word2vec_format(str(data_root/'vi-model-CBOW.bin'),binary=True)
sequence_rows=[]; oov_rows=[]; truncation_examples=[]; per_sample=[]
for dataset in ['Base','Augmented']:
    frame=raw[dataset]
    frequencies=Counter(word for text in frame.cbow_text for word in text.split())
    vocabulary={word for word,count in sorted(frequencies.items(),key=lambda p:(-p[1],p[0]))[:9998]}
    tokens=list(frequencies); total=sum(frequencies.values())
    covered=sum(count for word,count in frequencies.items() if word in vectors)
    oov_rows.extend(dict(dataset=dataset,word=word,count=count) for word,count in frequencies.most_common() if word not in vectors)
    print(dataset,'CBOW pretrained token coverage',covered/total,'type coverage',sum(w in vectors for w in tokens)/len(tokens))
    cbow_tokens=frame.cbow_text.str.split().tolist()
    bpe_tokens=[tokenizer.encode(text,add_special_tokens=True,truncation=False) for text in frame.phobert_text]
    for position,row in enumerate(frame.itertuples()):
        words=cbow_tokens[position]
        per_sample.append(dict(dataset=dataset,sample_id=row.sample_id,label=row.label,cbow_length=len(words),bpe_length=len(bpe_tokens[position]),pretrained_oov_fraction=sum(w not in vectors for w in words)/max(1,len(words)),vocab_unk_fraction=sum(w not in vocabulary for w in words)/max(1,len(words))))
    for embedder,sequences,limits in [('CBOW',cbow_tokens,[128,256,300,512]),('PhoBERT',bpe_tokens,[128,256])]:
        sizes=np.array([len(x) for x in sequences])
        for limit in limits:
            # BPE gồm special tokens: phần nội dung bị mất là length - limit.
            removed=np.maximum(0,sizes-limit)
            sequence_rows.append(dict(dataset=dataset,embedder=embedder,limit=limit,median=float(np.median(sizes)),p90=float(np.quantile(sizes,.9)),p95=float(np.quantile(sizes,.95)),p99=float(np.quantile(sizes,.99)),maximum=int(sizes.max()),truncated_documents=int((removed>0).sum()),truncated_percent=100*float((removed>0).mean()),tokens_removed=int(removed.sum()),tokens_removed_percent=100*float(removed.sum()/max(1,sizes.sum()))))
            if (embedder=='CBOW' and limit==300) or (embedder=='PhoBERT' and limit==256):
                candidates=frame.assign(size=sizes).loc[sizes>limit].sort_values('sample_id').head(4)
                for row in candidates.itertuples():
                    index=frame.index.get_loc(row.Index)
                    sequence=sequences[index]
                    lost=' '.join(sequence[limit:]) if embedder=='CBOW' else tokenizer.decode(sequence[limit-1:-1])
                    truncation_examples.append(dict(dataset=dataset,embedder=embedder,sample_id=row.sample_id,text=row.text,length=row.size,lost_content=lost))
sequence=pd.DataFrame(sequence_rows); display(sequence); sequence.to_csv(OUT/'sequence_limits.csv',index=False)
samples=pd.DataFrame(per_sample); samples.to_csv(OUT/'sample_token_diagnostics.csv',index=False)
oov=pd.DataFrame(oov_rows); oov.to_csv(OUT/'cbow_oov.csv',index=False); display(oov.groupby('dataset').head(20))
pd.DataFrame(truncation_examples,columns=['dataset','embedder','sample_id','text','length','lost_content']).to_csv(OUT/'truncation_examples.csv',index=False); display(pd.DataFrame(truncation_examples))
fig,axes=plt.subplots(1,2,figsize=(12,4))
for ax,column in zip(axes,['cbow_length','bpe_length']):
    for dataset,part in samples.groupby('dataset'):
        values=np.sort(part[column]); ax.step(values,100*np.arange(1,len(values)+1)/len(values),label=dataset)
    ax.axvline(300 if column=='cbow_length' else 256,color='red',linestyle='--'); ax.set(xlabel=column,ylabel='% câu không vượt ngưỡng'); ax.legend()
fig.tight_layout(); fig.savefig(OUT/'token-limits.png',dpi=140); plt.show(); plt.close(fig)
fig,axes=plt.subplots(1,2,figsize=(12,4))
for ax,col in zip(axes,['pretrained_oov_fraction','vocab_unk_fraction']):
    samples.boxplot(column=col,by='dataset',ax=ax); ax.set_title(col)
fig.suptitle('CBOW: OOV pretrained và UNK vocabulary là hai vấn đề khác nhau'); fig.tight_layout(); fig.savefig(OUT/'cbow-coverage.png',dpi=140); plt.show(); plt.close(fig)
# Kiểm tra cả khác biệt coverage trước/sau từng quy tắc trên cùng câu.
coverage=[]
for dataset in ['Base','Augmented']:
    frame=raw[dataset]
    for rule,fn in transforms.items():
        texts=frame.text.map(fn)
        words=texts.str.lower().map(ViTokenizer.tokenize).str.split()
        counts=Counter(w for sentence in words for w in sentence)
        bpe_sizes=texts.map(segment).map(lambda s:len(tokenizer.encode(s,add_special_tokens=True,truncation=False)))
        coverage.append(dict(dataset=dataset,rule=rule,cbow_tokens=sum(counts.values()),cbow_oov_tokens=sum(c for w,c in counts.items() if w not in vectors),cbow_oov_types=sum(w not in vectors for w in counts),bpe_total=int(bpe_sizes.sum()),bpe_over256=int(bpe_sizes.gt(256).sum())))
coverage=pd.DataFrame(coverage); coverage.to_csv(OUT/'embedder_rule_comparison.csv',index=False); display(coverage)
print('Không có classifier được huấn luyện; thống kê embedder chỉ dùng Base/Augmented.')
import zipfile, platform
manifest={'dataset':DATASET_HANDLE,'cpu_only':True,'test_usage':'schema and text overlap only','sources':{name:hashlib.sha256(path.read_bytes()).hexdigest() for name,path in paths.items()},'python':platform.python_version()}
(OUT/'source.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2))
with zipfile.ZipFile('/content/vlsp-eda-complete.zip','w',zipfile.ZIP_DEFLATED) as archive:
    for path in OUT.iterdir():
        if path.is_file():archive.write(path,path.name)
print('EDA_COMPLETE',OUT.resolve())
